# Visualizations

Use this notebook for saved result plots and slower exploratory visualizations. All figure saves should use `save_figure`, which removes titles before export.


In [ ]:
from collections import defaultdict
from pathlib import Path
import itertools

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from qiskit.quantum_info import state_fidelity

from broadcasting import load_run, list_runs
from broadcasting.plotting import plot_3d_fidelity, plot_run_sweep, save_figure
from broadcasting.simulation import run_broadcast_no_qec, run_broadcast_qec
from broadcasting.validation import dedupe_by_job, find_duplicate_jobs, group_by_cohort

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})

RESULTS_DIR = Path("results")
FIGURE_DIR = Path("figures")
SAVE_FIGURES = False


## Load Runs


In [ ]:
runs = list_runs(RESULTS_DIR)
print(f"Found {len(runs)} run(s) in {RESULTS_DIR}/")
for i, run in enumerate(runs):
    sweep = run.get("sweep", {})
    qec = "QEC" if run.get("use_qec") else "no-QEC"
    opt = run.get("optimization_level")
    opt_text = f" opt={opt}" if opt is not None else ""
    print(
        f"[{i:>2}] {run['filename']}: M={run['M']} N={run['N']} {qec} "
        f"{run['experiment_type']}/{run.get('backend')}{opt_text} "
        f"{len(sweep.get('values', []))} {sweep.get('axis', '?')}-points"
    )


## Plot One Saved Run


In [ ]:
RUN_INDEX = -1

if not runs:
    print("No saved runs to plot.")
else:
    run = runs[RUN_INDEX]
    tau_kwargs = (
        {"tau_scale": 4e-3, "tau_label": "Idle delay (us)"}
        if run.get("sweep", {}).get("axis") == "tau"
        else {}
    )
    fig = plot_run_sweep(run, show=False, **tau_kwargs)
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / f"{Path(run['filename']).stem}.png")
    plt.show()


## Compare Saved Runs


In [ ]:
selected = [run for run in runs if run["experiment_type"] == "hardware"]
# selected = [run for run in runs if run.get("use_qec")]
# selected = [runs[0], runs[3]]

if not selected:
    print("No runs matched the selection.")
else:
    ncols = min(len(selected), 3)
    nrows = int(np.ceil(len(selected) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.2 * nrows), squeeze=False, sharey=True)

    for idx, run in enumerate(selected):
        ax = axes[idx // ncols][idx % ncols]
        tau_kwargs = (
            {"tau_scale": 4e-3, "tau_label": "Idle delay (us)"}
            if run.get("sweep", {}).get("axis") == "tau"
            else {}
        )
        plot_run_sweep(run, ax=ax, show=False, **tau_kwargs)
        ax.text(
            0.02,
            0.04,
            f"M={run['M']} N={run['N']}\n{run.get('backend')}",
            transform=ax.transAxes,
            fontsize=8,
            va="bottom",
            ha="left",
            bbox={"facecolor": "white", "alpha": 0.75, "edgecolor": "none"},
        )

    for idx in range(len(selected), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "selected_runs.png")
    plt.show()


## Hardware Fidelity At Tau Zero


In [ ]:
hardware_runs = [run for run in runs if run["experiment_type"] == "hardware"]
hardware_runs = dedupe_by_job(hardware_runs)

# Stratify by (backend, shots) instead of silently pooling every hardware run --
# these are not a controlled N-scaling sweep on one device (see ACTION_PLAN.md
# Phase 3, item 39): report the true cohort composition explicitly.
cohorts = group_by_cohort(hardware_runs, keys=("backend", "shots"))
print("Hardware run cohorts (backend, shots) -> count:")
for key, group in sorted(cohorts.items(), key=lambda kv: str(kv[0])):
    print(f"  {key}: {len(group)} run(s)")

points = []  # (M, N, backend, mean_fid, worst_fid, min_fid, max_fid)
for run in hardware_runs:
    sweep_values = np.asarray(run.get("sweep", {}).get("values", []), dtype=float)
    if sweep_values.size == 0:
        continue
    zero_index = int(np.argmin(np.abs(sweep_values)))
    if abs(sweep_values[zero_index]) > 1e-9:
        # Nearest point isn't actually tau=0 -- don't silently mislabel it.
        print(f"  Skipping {run['filename']}: nearest sweep value is {sweep_values[zero_index]}, not 0.")
        continue
    fids = np.asarray(run["fidelities"][zero_index], dtype=float)
    points.append((run["M"], run["N"], run.get("backend"), float(fids.mean()), float(fids.min()), float(fids.min()), float(fids.max())))

if not points:
    print("No hardware tau-zero data found.")
else:
    fig, ax = plt.subplots(figsize=(7, 5))

    m_values = sorted({p[0] for p in points})
    backend_values = sorted({p[2] for p in points})
    # Colorblind-safe qualitative palette (tab10) keyed by M, instead of a
    # sequential colormap (magma) that gives low contrast between adjacent M.
    m_colors = {m: c for m, c in zip(m_values, plt.cm.tab10(np.linspace(0, 1, max(len(m_values), 2))))}
    backend_markers = {b: mk for b, mk in zip(backend_values, ["o", "s", "^", "D", "v", "P"])}

    seen_labels = set()
    n_values = sorted({p[1] for p in points})
    for M, N, backend, mean_fid, worst_fid, min_fid, max_fid in sorted(points):
        label = f"M={M}, {backend}"
        show_label = label not in seen_labels
        seen_labels.add(label)
        # Full spread (min-max across receivers) as an error bar, worst-case
        # receiver fidelity as a separate marker -- not just the mean.
        ax.errorbar(
            N, mean_fid,
            yerr=[[mean_fid - min_fid], [max_fid - mean_fid]],
            fmt=backend_markers[backend], color=m_colors[M],
            markeredgecolor="black", markersize=7, capsize=4, linewidth=1,
            label=label if show_label else "_nolegend_",
        )
        ax.scatter(N, worst_fid, color=m_colors[M], marker="_", s=120, linewidths=2)

    ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Number of receivers (N)")
    ax.set_ylabel("Receiver fidelity at tau=0 (mean, spread, worst-case \u2014)")
    ax.set_ylim(0, 1.05)
    # Compressed x-axis: integer N ticks only, tight margins instead of the
    # default padded continuous axis.
    ax.set_xticks(n_values)
    ax.set_xlim(min(n_values) - 0.5, max(n_values) + 0.5)
    ax.legend(title="Senders, backend", fontsize=8)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "hardware_tau0_scaling.png")
        save_figure(fig, Path("manuscript") / "fidelity scaling hardware.png")
    plt.show()



## QEC Crossover From Saved Simulation Runs


In [ ]:
def first_matching_run(*, M, N, use_qec, backend):
    for run in runs:
        if (
            run["experiment_type"] == "simulation"
            and run["M"] == M
            and run["N"] == N
            and bool(run.get("use_qec")) == use_qec
            and run.get("backend") == backend
            and run.get("sweep", {}).get("axis") == "p"
        ):
            return run
    return None

exact_no_qec = first_matching_run(M=1, N=2, use_qec=False, backend="aer_exact")
exact_qec = first_matching_run(M=1, N=2, use_qec=True, backend="aer_exact")
sampled_qec = first_matching_run(M=1, N=2, use_qec=True, backend="aer_sampling")

if not all([exact_no_qec, exact_qec, sampled_qec]):
    print("Need saved M=1, N=2 exact no-QEC, exact QEC, and sampled QEC p-sweeps.")
else:
    p = np.asarray(exact_no_qec["sweep"]["values"], dtype=float)
    curves = {
        "Exact no QEC": np.asarray(exact_no_qec["fidelities"], dtype=float).mean(axis=1),
        "Exact with QEC": np.asarray(exact_qec["fidelities"], dtype=float).mean(axis=1),
        "Sampling with QEC": np.asarray(sampled_qec["fidelities"], dtype=float).mean(axis=1),
    }

    fig, ax = plt.subplots(figsize=(8, 5))
    for label, values in curves.items():
        ax.plot(p, values, label=label)
    ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Depolarizing probability p")
    ax.set_ylabel("Average receiver fidelity")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "qec_crossover.png")
        save_figure(fig, Path("manuscript") / "qec vs no qec vs sampling.png")
    plt.show()


## Exact Vs Sampling Groups


In [ ]:
groups = defaultdict(list)
for run in runs:
    if run["experiment_type"] == "simulation" and run.get("use_qec") and run.get("sweep", {}).get("axis") == "p":
        groups[(run["M"], run["N"])].append(run)

if not groups:
    print("No QEC simulation runs found.")
else:
    ncols = min(len(groups), 3)
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.2 * nrows), squeeze=False, sharey=True)

    for idx, ((M, N), group) in enumerate(sorted(groups.items())):
        ax = axes[idx // ncols][idx % ncols]
        for run in group:
            p = np.asarray(run["sweep"]["values"], dtype=float)
            avg = np.asarray(run["fidelities"], dtype=float).mean(axis=1)
            if run.get("backend") == "aer_exact":
                ax.plot(p, avg, color="tab:red", linewidth=1.8, label="exact")
            elif run.get("backend") == "aer_sampling":
                ax.plot(p, avg, "--", color="tab:blue", alpha=0.45, label=f"sampling {run.get('n_samples')}")
        ax.text(0.03, 0.05, f"M={M}, N={N}", transform=ax.transAxes, fontsize=9)
        ax.axhline(0.5, color="gray", linestyle=":", alpha=0.4)
        ax.set_xlabel("Depolarizing probability p")
        if idx % ncols == 0:
            ax.set_ylabel("Average fidelity")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)

    for idx in range(len(groups), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "exact_sampling_groups.png")
    plt.show()


## Pairwise Receiver Exploration

This cell contains optional pairwise-fidelity exploration. It is off by default because the QEC contour can be slow.


In [ ]:
RUN_PAIRWISE_SCATTER = False
RUN_PAIRWISE_CONTOUR = False

if RUN_PAIRWISE_SCATTER:
    M_pair, N_pair = 2, 3
    repetitions = 2
    outcomes = list(itertools.product(range(N_pair + 1), repeat=M_pair))
    results = []
    for _ in range(repetitions):
        for outcome in outcomes:
            p_list_pair = np.random.uniform(0.0, 1.0, N_pair)
            theta_pair = np.random.uniform(0.0, 2 * np.pi, M_pair)
            results.append(
                run_broadcast_qec(
                    M=M_pair,
                    N=N_pair,
                    alpha=1 / np.sqrt(2),
                    theta_list=theta_pair,
                    p_list=p_list_pair,
                    outcomes_list=outcome,
                )
            )

    n = len(results)
    points = np.zeros((3, 3 * n))
    for i, (fidelities, _, reduced_states) in enumerate(results):
        pairs = [(0, 1), (0, 2), (1, 2)]
        for j, (a, b) in enumerate(pairs):
            points[0, i + j * n] = fidelities[a]
            points[1, i + j * n] = fidelities[b]
            points[2, i + j * n] = state_fidelity(reduced_states[a], reduced_states[b])

    fig = plot_3d_fidelity(points, show=False)
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "pairwise_receiver_scatter.png")
    plt.show()

if RUN_PAIRWISE_CONTOUR:
    M_c, N_c = 1, 2
    theta_c = [np.pi / 5]
    p_grid = np.linspace(0.0, 1.0, 18)
    outcomes = list(itertools.product(range(N_c + 1), repeat=M_c))

    def collect_points(use_qec):
        runner = run_broadcast_qec if use_qec else run_broadcast_no_qec
        F1, F2, F12 = [], [], []
        for p1 in p_grid:
            for p2 in p_grid:
                f1 = f2 = f12 = 0.0
                for outcome in outcomes:
                    fidelities, _, reduced = runner(
                        M=M_c,
                        N=N_c,
                        alpha=1 / np.sqrt(2),
                        theta_list=theta_c,
                        p_list=[p1, p2],
                        outcomes_list=outcome,
                    )
                    f1 += fidelities[0]
                    f2 += fidelities[1]
                    f12 += state_fidelity(reduced[0], reduced[1])
                denom = len(outcomes)
                F1.append(f1 / denom)
                F2.append(f2 / denom)
                F12.append(f12 / denom)
        return np.asarray(F1), np.asarray(F2), np.asarray(F12)

    datasets = [collect_points(False), collect_points(True)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
    for ax, (F1, F2, F12), label in zip(axes, datasets, ["No QEC", "QEC"]):
        tcf = ax.tricontourf(F1, F2, F12, levels=np.linspace(0, 1, 21), cmap=cm.magma, vmin=0, vmax=1)
        ax.text(0.03, 0.05, label, transform=ax.transAxes, color="white", fontsize=10)
        ax.set_xlabel("Fidelity(receiver 1, target)")
        ax.set_ylabel("Fidelity(receiver 2, target)")
        ax.set_xlim(0.5, 1.0)
        ax.set_ylim(0.5, 1.0)
        ax.set_aspect("equal")
        fig.colorbar(tcf, ax=ax, label="Fidelity(receiver 1, receiver 2)")
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "pairwise_receiver_contours.png")
    plt.show()


## Joint/Global Fidelity And Receiver Covariance (No New Hardware Time Needed)

Every hardware run already stores the full joint receiver readout counts, so global (all-receivers)
fidelity, worst-receiver fidelity, and pairwise receiver success-covariance can all be computed
directly from existing saved runs -- no new hardware jobs needed. Set `RUN_FILTER`/`SWEEP_INDEX`
below to pick which saved run and sweep point (e.g. `tau=0`) to analyze.


In [ ]:
RUN_FILENAME = None  # e.g. "run_20260518_120232.json"; None -> first hardware run with counts
SWEEP_INDEX = 0      # index into the sweep axis (0 is usually tau=0)
THETA_INDEX = 0      # which theta sample, if more than one was recorded


def _joint_stats(counts: dict[str, int], N: int) -> dict[str, float]:
    total = sum(counts.values())
    local = [sum(c for bs, c in counts.items() if bs[N - 1 - i] == "0") / total for i in range(N)]
    global_fid = sum(c for bs, c in counts.items() if bs == "0" * N) / total
    product_of_locals = float(np.prod(local))

    cov = {}
    for i in range(N):
        for j in range(i + 1, N):
            xi = np.array([1.0 if bs[N - 1 - i] == "0" else 0.0 for bs in counts for _ in range(counts[bs])])
            xj = np.array([1.0 if bs[N - 1 - j] == "0" else 0.0 for bs in counts for _ in range(counts[bs])])
            cov[f"({i},{j})"] = float(np.cov(xi, xj)[0, 1])

    return {
        "mean_local_fidelity": float(np.mean(local)),
        "worst_receiver_fidelity": float(np.min(local)),
        "global_fidelity": global_fid,
        "product_of_locals": product_of_locals,
        "pairwise_covariance": cov,
    }


candidates = [
    r for r in runs
    if r["experiment_type"] == "hardware" and r.get("counts")
    and (RUN_FILENAME is None or r["filename"] == RUN_FILENAME)
]
if not candidates:
    print("No hardware run with saved joint counts found.")
else:
    run = candidates[0]
    counts = run["counts"][THETA_INDEX][SWEEP_INDEX]
    stats = _joint_stats(counts, run["N"])
    print(f"Run: {run['filename']} (backend={run.get('backend')}, N={run['N']}, "
          f"sweep[{SWEEP_INDEX}]={run['sweep']['values'][SWEEP_INDEX]})")
    for key, value in stats.items():
        print(f"  {key}: {value}")
    print(
        "\nNote: pairwise covariance != 0 is necessary but not sufficient evidence of "
        "correlated physical noise -- shared preparation/feedforward errors, readout "
        "crosstalk, and pooling different thetas/jobs can also produce nonzero covariance."
    )
